In [1]:
#installing cassandra driver to use cassandra
#pyspark - enables reading and cleaning of large JSON data using spark data frames
#cassandra driver - enables python to connect to cassandra by running CQL commands
#selecting pyspark version for compatability issues

!pip install pyspark==3.5.1
!pip install cassandra-driver
!pip install pandas

#***run to run queries***

In [2]:
import sys, os

from cassandra.cluster import Cluster
from cassandra.concurrent import execute_concurrent_with_args

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.functions import (
    round, col, trim, regexp_replace, to_timestamp, from_unixtime, when, collect_set, avg, count, sum, lit
)

#for timing queries
import time
import pandas as pd


concurrency = 2 # can change this if docker is slow/fails to 100
batch_size = 500  #can change to 800

#***run to run queries***

In [3]:
#this creates a casandra cluster and connects the cassandra session - it then runs a small query to test the connection as i was having connection issues.

CASSANDRA_HOST = "127.0.0.1"
cluster = Cluster([CASSANDRA_HOST])
session = cluster.connect()

version = session.execute("SELECT release_version FROM system.local").one().release_version
#print("Connected, Version:", version)

#***run to run queries***


In [4]:
keyspace = "amazon_reviews"

# prints keyspaces
# print("Keyspaces:")
#for r in session.execute("SELECT keyspace_name FROM system_schema.keyspaces"):
    #print("-", r.keyspace_name)

# prints tables that are in the keyspace
#print(f"\nTables in {keyspace}:")
#for r in session.execute(
    #"SELECT table_name FROM system_schema.tables WHERE keyspace_name=%s",
    #(keyspace,)
#):
    #print("-", r.table_name)

#***run to run queries***


In [5]:
#set up notes

#you need to have cassandra running in docker - default host 127.0.0.1, port 9042

#change file paths if you download on your device - mine were:
# project_folder/ notebook.ipynb/data/cleaned_reviews.jsonl/cleaned_meta.jsonl

#there are 5 queries at the bottom to test the 5 tables in the cassandra database

#the 6 tables are:

#products - product metadata one row per product
#reviews by product - reviews grouped by product, ordered in time with the most recent being first
#reviews by user - all reviews written by user again in order and most recent first. you use the user id as the primary key to search
#product ratings - average rating and rating counts per product
#product keywords - you can search a product using keywords. the partition key is the keywords and then clustering key is the parent_asin
#all data - all data in one table

#The 5 Queries are:

#Products table:
#Returns product metadata – ASIN, Title and Description.

#Reviews by product:
#Gets a parent ASIN and retreives 5 reviews for that product and returns: Review time, User ID, Rating, Review title

#Reviews by user:
#selects user_id and retrieves 5 written by that user, it returns the user ID, review time, product asin, rating and review title.

#Product Ratings:
#Looks up a specific product – B00Z03RC80 and returns all the data for ratings per product

#Product Keywords :
#Keyword “hair” which returns parent_asin and title


#printing the paths for the environment and where python is working from for debugging purposes.


#DEBUGGING
#print("Python interpreter Path:", sys.executable)
#print("Directory Path:", os.getcwd())


In [6]:
#starts the spark session so that the JSONL file can be written into spark data frames to be processed.  defines port as i was getting port clashes

spark = (
    SparkSession.builder
    .appName("SCC454-Amazon")
    .config("spark.ui.port", "4050")
    .config("spark.driver.memory","4g")
    .config("spark.executor.memory","4g")
    .getOrCreate()
)

#DEBUGGING
#print("spark version:", spark.version)
#print("master:", spark.sparkContext.master)


creating tables 1-6

In [7]:
# Creating keyspace tables
session.execute(f"""
CREATE KEYSPACE IF NOT EXISTS {keyspace}
WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': 1}};
""")

session.execute(f"USE {keyspace};")

#the 5 required tables for queries are created in cassandra here if they dont already exist from previous runs

# products table - product info by parent_asin
session.execute("""
CREATE TABLE IF NOT EXISTS products (
  parent_asin text PRIMARY KEY,
  title text,
  description text
);
""")

# reviews_by_product table - recent reviews per product - query by product and time
session.execute("""
CREATE TABLE IF NOT EXISTS reviews_by_product (
  parent_asin text,
  review_time timestamp,
  timestamp_convert text,
  user_id text,
  rating double,
  title text,
  text text,
  verified_purchase boolean,
  PRIMARY KEY ((parent_asin), review_time, user_id)
) WITH CLUSTERING ORDER BY (review_time DESC);
""")

# reviews_by_user table - all reviews by a user - query by user and time
session.execute("""
CREATE TABLE IF NOT EXISTS reviews_by_user (
  user_id text,
  review_time timestamp,
  timestamp_convert text,
  parent_asin text,
  rating double,
  title text,
  text text,
  verified_purchase boolean,
  PRIMARY KEY ((user_id), review_time, parent_asin)
) WITH CLUSTERING ORDER BY (review_time DESC);
""")

# product_ratings: stats by product
session.execute("""
CREATE TABLE IF NOT EXISTS product_ratings (
  parent_asin text PRIMARY KEY,
  avg_rating double,
  review_count bigint,
  count_1 bigint,
  count_2 bigint,
  count_3 bigint,
  count_4 bigint,
  count_5 bigint
);
""")

# product_keywords: keyword search support (keyword -> products)
session.execute("""
CREATE TABLE IF NOT EXISTS product_keywords (
  keyword text,
  parent_asin text,
  title text,
  PRIMARY KEY ((keyword), parent_asin)
);
""")

print(" Tables created/ already existed")

#6th table with full data set - denormalised, each row contains product information and the reviews, adds indexes

session.execute("""
CREATE TABLE IF NOT EXISTS all_data (
  parent_asin text,
  review_time timestamp,
  timestamp_convert text,
  user_id text,
  rating int,
  title text,
  text text,
  verified_purchase boolean,

  product_title text,
  product_description text,

  avg_rating double,
  review_count bigint,
  count_1 bigint,
  count_2 bigint,
  count_3 bigint,
  count_4 bigint,
  count_5 bigint,

  keywords set<text>,

  PRIMARY KEY ((parent_asin), review_time, user_id)
) WITH CLUSTERING ORDER BY (review_time DESC);
""")

# indexes to allow for queries
session.execute("CREATE INDEX IF NOT EXISTS all_data_user_id_idx ON amazon_reviews.all_data (user_id);")
session.execute("CREATE INDEX IF NOT EXISTS all_data_keywords_idx ON amazon_reviews.all_data (keywords);")

#CLEARS ALL TABLES BEFORE INSERT - REMOVE IF DONT WANT TO REMOVE DATA IN TABLLES!!! ***********
session.execute("TRUNCATE amazon_reviews.reviews_by_product;")
session.execute("TRUNCATE amazon_reviews.reviews_by_user;")
session.execute("TRUNCATE amazon_reviews.product_ratings;")
session.execute("TRUNCATE amazon_reviews.products;")
session.execute("TRUNCATE amazon_reviews.product_keywords;")
session.execute("TRUNCATE amazon_reviews.all_data;")
print("all tables CLEARED")

 Tables created/ already existed
all tables CLEARED


Inserting data into tables 1-6

In [8]:
#debugging

#rows = session.execute("""
#SELECT table_name FROM system_schema.tables
#WHERE keyspace_name='amazon_reviews';
#""")
#print("tables:", [row.table_name for row in rows])


In [9]:
#creating file path for cleaned data
data = r"C:\Users\Egome\PycharmProjects\SCC454_Amazon\data"
cleaned_reviews_path = os.path.join(data, "cleaned_reviews.jsonl")

#reading reviews data
reviews = spark.read.json(cleaned_reviews_path).repartition(200).cache()
#checking column names
reviews.count()
reviews.printSchema()
print(reviews.columns)
reviews.show(5, truncate=False)




root
 |-- helpful_vote: long (nullable = true)
 |-- month: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- timestamp_convert: string (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- year: long (nullable = true)

['helpful_vote', 'month', 'parent_asin', 'rating', 'text', 'timestamp', 'timestamp_convert', 'title', 'user_id', 'verified_purchase', 'year']
+------------+-----+-----------+------+------------------------------------------------------------------------------------------------------------------+-------------+-----------------------------+-------------+----------------------------+-----------------+----+
|helpful_vote|month|parent_asin|rating|text                                                                                                      

In [10]:
#inserting product and user reviews into the two tables created earlier.

insert_by_product = session.prepare(f"""
INSERT INTO {keyspace}.reviews_by_product
(parent_asin, review_time, timestamp_convert, user_id, rating, title, text, verified_purchase)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

insert_by_user = session.prepare(f"""
INSERT INTO {keyspace}.reviews_by_user
(user_id, review_time, timestamp_convert, parent_asin, rating, title, text, verified_purchase)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

#inserts in batches (faster than tolocaliterator)

def review_batches(df, batch_size=batch_size):
    batch_prod = []
    batch_user = []
    for r in df.toLocalIterator():
        batch_prod.append((
            r["parent_asin"], r["timestamp"], r["timestamp_convert"], r["user_id"], r["rating"],
            r["title"], r["text"], r["verified_purchase"]
        ))
        batch_user.append((
            r["user_id"], r["timestamp"], r["timestamp_convert"], r["parent_asin"], r["rating"],
            r["title"], r["text"], r["verified_purchase"]
        ))
        if len(batch_prod) >= batch_size:
            yield batch_prod, batch_user
            batch_prod, batch_user = [], []
    if batch_prod:
        yield batch_prod, batch_user

total = 0
for b_prod, b_user in review_batches(reviews):
    execute_concurrent_with_args(session, insert_by_product, b_prod, concurrency=concurrency)
    execute_concurrent_with_args(session, insert_by_user, b_user, concurrency=concurrency)
    total += len(b_prod)
    if total % 10000 == 0:
        print("n of data inserted so far:", total)

print("inserted", total)

#slower way row by row used when limited to 5000
#n = 0
#for row in reviews.toLocalIterator():
    #session.execute(insert_by_product, (
       # row["parent_asin"], row["timestamp"], row["user_id"], #row["rating"],
        #row["title"], row["text"], row["verified_purchase"]
    #))

    #session.execute(insert_by_user, (
        #row["user_id"], row["timestamp"], row["parent_asin"], #row["rating"],
       # row["title"], row["text"], row["verified_purchase"]
    #))

    #n += 1
    #if n % 500 == 0:
        #print("n of data inserted so far:", n)

#print("Finished inserting:", n, "is the final number of rows inserted")



n of data inserted so far: 10000
n of data inserted so far: 20000
n of data inserted so far: 30000
n of data inserted so far: 40000
n of data inserted so far: 50000
n of data inserted so far: 60000
n of data inserted so far: 70000
n of data inserted so far: 80000
n of data inserted so far: 90000
n of data inserted so far: 100000
n of data inserted so far: 110000
n of data inserted so far: 120000
n of data inserted so far: 130000
n of data inserted so far: 140000
n of data inserted so far: 150000
n of data inserted so far: 160000
n of data inserted so far: 170000
n of data inserted so far: 180000
n of data inserted so far: 190000
n of data inserted so far: 200000
n of data inserted so far: 210000
n of data inserted so far: 220000
n of data inserted so far: 230000
n of data inserted so far: 240000
n of data inserted so far: 250000
n of data inserted so far: 260000
n of data inserted so far: 270000
n of data inserted so far: 280000
n of data inserted so far: 290000
n of data inserted so f

In [11]:
#creates statistics per product in spark and groups the reviews by parent_asin. calculates average rating, total review count, counts number of 1,2,3,4,5 star ratings to produce a products stats dataframe.


product_stats = (
    reviews
    .groupBy("parent_asin")
    .agg(
        avg("rating").alias("avg_rating"),
        count("*").alias("review_count"),
        sum(when(col("rating") == 1, 1).otherwise(0)).alias("count_1"),
        sum(when(col("rating") == 2, 1).otherwise(0)).alias("count_2"),
        sum(when(col("rating") == 3, 1).otherwise(0)).alias("count_3"),
        sum(when(col("rating") == 4, 1).otherwise(0)).alias("count_4"),
        sum(when(col("rating") == 5, 1).otherwise(0)).alias("count_5"),
    )
)



In [12]:
insert_product_stats = session.prepare("""
INSERT INTO amazon_reviews.product_ratings
(parent_asin, avg_rating, review_count, count_1, count_2, count_3, count_4, count_5)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""")

#insert data into table in batches
def stats_batches(df, batch_size=batch_size):
    batch = []
    for r in df.toLocalIterator():
        batch.append((
            r["parent_asin"],
            float(r["avg_rating"]),
            int(r["review_count"]),
            int(r["count_1"]),
            int(r["count_2"]),
            int(r["count_3"]),
            int(r["count_4"]),
            int(r["count_5"]),
        ))
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

total = 0
for batch in stats_batches(product_stats):
    execute_concurrent_with_args(session, insert_product_stats, batch, concurrency=concurrency)
    total += len(batch)
    if total % 10000 == 0:
        print("n of data inserted so far:", total)

print("inserted", total)


#slow way insert row by row
#n = 0
#for row in product_stats.toLocalIterator():
   # session.execute(insert_product_stats, (
    #    row["parent_asin"],
     #   float(row["avg_rating"]),
     #   int(row["review_count"]),
      #  int(row["count_1"]),
       # int(row["count_2"]),
       # int(row["count_3"]),
    # int(row["count_4"]),
       # int(row["count_5"]),
    #))
   # n += 1



n of data inserted so far: 10000
n of data inserted so far: 20000
n of data inserted so far: 30000
n of data inserted so far: 40000
n of data inserted so far: 50000
n of data inserted so far: 60000
n of data inserted so far: 70000
n of data inserted so far: 80000
n of data inserted so far: 90000
n of data inserted so far: 100000
n of data inserted so far: 110000
inserted 112565


In [13]:
#defines meta data file path

data_path = r"C:\Users\Egome\PycharmProjects\SCC454_Amazon\data"
cleaned_meta_path = os.path.join(data_path, "cleaned_metadata.jsonl")


In [14]:
#reading meta data
products_df = spark.read.json(cleaned_meta_path)
#viewing cleaned meta data columns
products_df.printSchema()
print(products_df.columns)
products_df.show(5, truncate=False)

root
 |-- Age Range (Description): string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Hair Type: string (nullable = true)
 |-- Item Form: string (nullable = true)
 |-- Material: string (nullable = true)
 |-- average_rating: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- description: string (nullable = true)
 |-- features_string: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating_number: long (nullable = true)
 |-- store: string (nullable = true)
 |-- title: string (nullable = true)

['Age Range (Description)', 'Brand', 'Hair Type', 'Item Form', 'Material', 'average_rating', 'categories', 'description', 'features_string', 'main_category', 'parent_asin', 'rating_number', 'store', 'title']
+-----------------------+--------+---------+---------+--------+--------------+----------+-----------------------------------------------------

In [15]:
#inserts data into products table

products_sample = (products_df)
#.limit(5000)) #**********remove if required!!!!

insert_product = session.prepare("""
INSERT INTO amazon_reviews.products (parent_asin, title, description)
VALUES (?, ?, ?)
""")

def products_batches(df, batch_size=batch_size):
    batch = []
    for r in df.toLocalIterator():
        batch.append((r["parent_asin"], r["title"], r["description"]))
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

total = 0
for batch in products_batches(products_sample):
    execute_concurrent_with_args(session, insert_product, batch, concurrency=concurrency)
    total += len(batch)
    if total % 10000 == 0:
        print("n of data inserted so far:", total)

print("inserted into products:", total)


#slow way used when inserted 5000 row by row
#n = 0
#for row in products_sample.toLocalIterator():
    #.execute(insert_product, (row["parent_asin"], row["title"], #row["description"]))
    #n += 1



n of data inserted so far: 10000
n of data inserted so far: 20000
n of data inserted so far: 30000
n of data inserted so far: 40000
n of data inserted so far: 50000
n of data inserted so far: 60000
n of data inserted so far: 70000
n of data inserted so far: 80000
n of data inserted so far: 90000
n of data inserted so far: 100000
n of data inserted so far: 110000
inserted into products: 112590


In [16]:
#creating keyword index from products
#lowercase, remove punctuation, split into tokens, and put them into 1 row per token, filters junk words and removed duplicates
#inverted index created for cassandra keyword search

kw_df = (
    products_sample
    .select("parent_asin", "title")
    .withColumn("title_lc", F.lower(F.col("title")))
    .withColumn("title_lc", F.regexp_replace(F.col("title_lc"), r"[^a-z0-9\s]+", " "))
    .withColumn("tokens", F.split(F.col("title_lc"), r"\s+"))
    .withColumn("keyword", F.explode(F.col("tokens")))
    # remove junk words
    .filter(F.length(F.col("keyword")) >= 3)
    .filter(~F.col("keyword").rlike(r"^\d+$"))          # pure numbers
    .filter(~F.col("keyword").rlike(r"^\d+(oz|mm|ml)$"))# units
    .filter(~F.col("keyword").isin(
        "with","from","this","that","your","for","and","the",
        "are","was","were","you","all","each","set","pack"
    ))
    .select("keyword", "parent_asin", "title")
    .dropDuplicates(["keyword", "parent_asin"])
)




In [17]:
#inserting keywords into the keyword table

insert_kw = session.prepare("""
INSERT INTO amazon_reviews.product_keywords (keyword, parent_asin, title)
VALUES (?, ?, ?)
""")

def keywords_batches(df, batch_size=batch_size):
    batch = []
    for r in df.toLocalIterator():
        batch.append((r["keyword"], r["parent_asin"], r["title"]))
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

total = 0
for batch in keywords_batches(kw_df):
    execute_concurrent_with_args(session, insert_kw, batch, concurrency=concurrency)
    total += len(batch)
    if total % 10000 == 0:
        print("n of data inserted so far:", total)

print("inserted", total)


#slower way of inserting data row by row

#n = 0
#for row in kw_df.toLocalIterator():
    #session.execute(insert_kw, (row["keyword"], row["parent_asin"], row["title"]))
    #n += 1
    #if n % 2000 == 0:
        #print("Inserted number of rows so far:", n)

#print("finished Inserting clean keyword rows:", n, "number of rows inserted in total")


n of data inserted so far: 10000
n of data inserted so far: 20000
n of data inserted so far: 30000
n of data inserted so far: 40000
n of data inserted so far: 50000
n of data inserted so far: 60000
n of data inserted so far: 70000
n of data inserted so far: 80000
n of data inserted so far: 90000
n of data inserted so far: 100000
n of data inserted so far: 110000
n of data inserted so far: 120000
n of data inserted so far: 130000
n of data inserted so far: 140000
n of data inserted so far: 150000
n of data inserted so far: 160000
n of data inserted so far: 170000
n of data inserted so far: 180000
n of data inserted so far: 190000
n of data inserted so far: 200000
n of data inserted so far: 210000
n of data inserted so far: 220000
n of data inserted so far: 230000
n of data inserted so far: 240000
n of data inserted so far: 250000
n of data inserted so far: 260000
n of data inserted so far: 270000
n of data inserted so far: 280000
n of data inserted so far: 290000
n of data inserted so f

In [18]:

# keywords grouped per product as a set so matches cassandra set
kw_per_product = (
    kw_df
    .groupBy("parent_asin")
    .agg(collect_set("keyword").alias("keywords"))
)

# denormalised data frame - writing data into table
all_data_df = (
    reviews
    .join(products_sample.select("parent_asin", col("title").alias("product_title"), "description"), on="parent_asin", how="left")
    .join(product_stats, on="parent_asin", how="left")
    .join(kw_per_product, on="parent_asin", how="left")
    .select(
        col("parent_asin"),
        col("timestamp").alias("review_time"),
        col("timestamp_convert"),
        col("user_id"),
        col("rating"),
        col("title"),
        col("text"),
        col("verified_purchase"),
        col("product_title"),
        col("description").alias("product_description"),
        col("avg_rating"),
        col("review_count"),
        col("count_1"), col("count_2"), col("count_3"), col("count_4"), col("count_5"),
        col("keywords")
    )
)

In [19]:
#inserting denormalised data into table 6
insert_all = session.prepare("""
INSERT INTO amazon_reviews.all_data
(parent_asin, review_time, timestamp_convert, user_id, rating, title, text, verified_purchase,
 product_title, product_description,
 avg_rating, review_count, count_1, count_2, count_3, count_4, count_5,
 keywords)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

def all_data_batches(df, batch_size=batch_size):
    batch = []
    for r in df.toLocalIterator():
        batch.append((
            r["parent_asin"],
            r["review_time"],
            r["timestamp_convert"],
            r["user_id"],
            int(r["rating"]) if r["rating"] is not None else None,
            r["title"],
            r["text"],
            r["verified_purchase"],
            r["product_title"],
                r["product_description"],
            float(r["avg_rating"]) if r["avg_rating"] is not None else None,
            int(r["review_count"]) if r["review_count"] is not None else None,
            int(r["count_1"]) if r["count_1"] is not None else None,
            int(r["count_2"]) if r["count_2"] is not None else None,
            int(r["count_3"]) if r["count_3"] is not None else None,
            int(r["count_4"]) if r["count_4"] is not None else None,
            int(r["count_5"]) if r["count_5"] is not None else None,
            set(r["keywords"]) if r["keywords"] is not None else set(),
        ))
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

total = 0
for batch in all_data_batches(all_data_df):
    execute_concurrent_with_args(session, insert_all, batch, concurrency=concurrency)
    total += len(batch)
    if total % 10000. == 0:
        print("n of data inserted so far:", total)

print("inserted", total)


#n = 0
#for row in all_data_df.toLocalIterator():
    #session.execute(insert_all, (
 #       row["parent_asin"],
  #      row["timestamp"],
   #     row["user_id"],
    #    int(row["rating"]) if row["rating"] is not None else None,
     #   row["title"],
      #  row["text"],
       # row["verified_purchase"],
#
 #       row["product_title"],
  #      row["product_description"],
#
 #       float(row["avg_rating"]) if row["avg_rating"] is not None else None,
  #      int(row["review_count"]) if row["review_count"] is not None else None,
   #     int(row["count_1"]) if row["count_1"] is not None else None,
    #    int(row["count_2"]) if row["count_2"] is not None else None,
     #   int(row["count_3"]) if row["count_3"] is not None else None,
      #  int(row["count_4"]) if row["count_4"] is not None else None,
       # int(row["count_5"]) if row["count_5"] is not None else None,

        #set(row["keywords"]) if row["keywords"] is not None else set(),
    #))
    #n += 1
    # if n % 5000 == 0:
    #     print("inserted", n)

#print("Finished inserting data")

n of data inserted so far: 10000
n of data inserted so far: 20000
n of data inserted so far: 30000
n of data inserted so far: 40000
n of data inserted so far: 50000
n of data inserted so far: 60000
n of data inserted so far: 70000
n of data inserted so far: 80000
n of data inserted so far: 90000
n of data inserted so far: 100000
n of data inserted so far: 110000
n of data inserted so far: 120000
n of data inserted so far: 130000
n of data inserted so far: 140000
n of data inserted so far: 150000
n of data inserted so far: 160000
n of data inserted so far: 170000
n of data inserted so far: 180000
n of data inserted so far: 190000
n of data inserted so far: 200000
n of data inserted so far: 210000
n of data inserted so far: 220000
n of data inserted so far: 230000
n of data inserted so far: 240000
n of data inserted so far: 250000
n of data inserted so far: 260000
n of data inserted so far: 270000
n of data inserted so far: 280000
n of data inserted so far: 290000
n of data inserted so f

Queries for 5 individual tables

In [20]:
#1
# #testing keyword table by performing query

keyword = "hair"


start = time.time()
rows = session.execute(
    """
    SELECT parent_asin, title
    FROM amazon_reviews.product_keywords
    WHERE keyword=%s
    LIMIT 10
    """,
    (keyword,)
)
fetched = list(rows)
end = time.time()

keyword_time = end - start

print(f"Results for keyword='{keyword}':")
for row in fetched:
    print(row)


Results for keyword='hair':
Row(parent_asin='0989205339', title='inspire hair fashion salon client volume 92 hairstyle men bk v92')
Row(parent_asin='4293845755', title='wella hair color post pump')
Row(parent_asin='B000052WYR', title='nair hair remover 4 minute lotion aloe lanolin')
Row(parent_asin='B00009RB1B', title='philip norelco d350 dfiner precise facial hair shaper')
Row(parent_asin='B0000YUZIS', title='authentic revo styler rotating hair brush system')
Row(parent_asin='B0000YVA3M', title='xgentle surgi cream facial hair remover')
Row(parent_asin='B00011JN5G', title='american international industry clean easy deluxe electrolysis home coarse hair')
Row(parent_asin='B00016KML0', title='velcro hair roller 19 large volume size roller')
Row(parent_asin='B0001D358A', title='finishing touch personal hair remover')
Row(parent_asin='B00023K3WU', title='large victorian style sterling silver hair pin clip barrette stick dangling accent')


In [21]:
#2
# #testing product rating table by performing query

asin = "B00Z03RC80"

start = time.time()
row = session.execute(
    """
    SELECT *
    FROM amazon_reviews.product_ratings
    WHERE parent_asin=%s
    """,
    (asin,)
).one()
end = time.time()

product_rating_time = end - start

print(row)


None


In [22]:
#3
# #testing reviews by user by performing query

user = reviews.select("user_id").limit(1).collect()[0]["user_id"]

start = time.time()
rows = session.execute("""
SELECT user_id, timestamp_convert, parent_asin, rating, title
FROM amazon_reviews.reviews_by_user
WHERE user_id=%s
LIMIT 5
""", (user,))
fetched = list(rows)
end = time.time()

user_review_time = end - start

for row in fetched:
    print(row)

Row(user_id='AHS3BYEXVP4RMJMIDJUDKGFROX7Q', timestamp_convert='2018-05-02T02:44:10.116+01:00', parent_asin='B00U0Z0IUM', rating=5.0, title='love smell')


In [23]:
#4
# #testing reviews by product by performing query

product = reviews.select("parent_asin").limit(1).collect()[0]["parent_asin"]

start = time.time()
rows = session.execute("""
SELECT parent_asin, timestamp_convert, user_id, rating, title
FROM amazon_reviews.reviews_by_product
WHERE parent_asin=%s
LIMIT 5
""", (product,))
fetched = list(rows)
end = time.time()

product_review_time = end - start

for row in fetched:
    print(row)


Row(parent_asin='B00U0Z0IUM', timestamp_convert='2019-01-28T21:47:51.596Z', user_id='AE422FOZHQIMARXVYUW6W2DGXMBQ', rating=5.0, title='love')
Row(parent_asin='B00U0Z0IUM', timestamp_convert='2018-10-04T08:16:51.076+01:00', user_id='AGHOEUGLLBU4J554C6RG6RXL4SLQ', rating=5.0, title='awesome soap imo people like chemical sensitivity')
Row(parent_asin='B00U0Z0IUM', timestamp_convert='2018-05-15T00:35:43.523+01:00', user_id='AFQYT7XPMVNOZL57M54M7RRWUMQA', rating=1.0, title='wonderful scent changed')
Row(parent_asin='B00U0Z0IUM', timestamp_convert='2018-05-08T03:23:53.742+01:00', user_id='AHG5427H6WYRNG2A7YGCHO6NOPHQ', rating=1.0, title='new formula waste money')
Row(parent_asin='B00U0Z0IUM', timestamp_convert='2018-05-02T02:44:10.116+01:00', user_id='AHS3BYEXVP4RMJMIDJUDKGFROX7Q', rating=5.0, title='love smell')


In [24]:
#5
# #testing products

test_product = products_sample.select("parent_asin").limit(1).collect()[0]["parent_asin"]

start = time.time()
row = session.execute("""
SELECT parent_asin, title, description
FROM amazon_reviews.products
WHERE parent_asin=%s
""", (test_product,)).one()
end = time.time()

product_time = end - start

print(row)


Row(parent_asin='B01CUPMQZE', title='howard lc008 leather conditioner 8 ounce 4 pack', description='')


Queries for table 6 with all data

In [25]:
#1
# product keywords test
keyword = "hair"

start = time.time()
rows = session.execute("""
select parent_asin, product_title
from amazon_reviews.all_data
where keywords contains %s
limit 10
""", (keyword,))
fetched = list(rows)
end = time.time()

keyword_time_all = end - start

print(f"'{keyword}':")
for row in fetched:
    print(row)


'hair':
Row(parent_asin='B091YQMF3W', product_title='10 piece paisley headband twisted knot design boho hair band knot bandana headwear retro flower printed elastic wide hairband scrunchies accessory woman girl')
Row(parent_asin='B06XPQBFX8', product_title='scotch porter hydrating hair wash men sulfate free shampoo 8 oz')
Row(parent_asin='B06XPQBFX8', product_title='scotch porter hydrating hair wash men sulfate free shampoo 8 oz')
Row(parent_asin='B06XPQBFX8', product_title='scotch porter hydrating hair wash men sulfate free shampoo 8 oz')
Row(parent_asin='B06XPQBFX8', product_title='scotch porter hydrating hair wash men sulfate free shampoo 8 oz')
Row(parent_asin='B06XPQBFX8', product_title='scotch porter hydrating hair wash men sulfate free shampoo 8 oz')
Row(parent_asin='B06XPQBFX8', product_title='scotch porter hydrating hair wash men sulfate free shampoo 8 oz')
Row(parent_asin='B07TXB63WJ', product_title='micro loop hair')
Row(parent_asin='B07HC9J2ZF', product_title='9a water wave

In [26]:
#2
# #product rating test - using asin
asin = "B00Z03RC80"

start = time.time()
row = session.execute("""
select parent_asin, avg_rating, review_count, count_1, count_2, count_3, count_4, count_5
from amazon_reviews.all_data
where parent_asin=%s
limit 1
""", (asin,)).one()
end = time.time()

product_rating_time_all = end - start

print(row)


None


In [27]:
#3
# #reviews by user test
user = reviews.select("user_id").limit(1).collect()[0]["user_id"]

start = time.time()
rows = session.execute("""
select user_id, review_time, parent_asin, rating, title
from amazon_reviews.all_data
where user_id=%s
limit 5
""", (user,))
fetched = list(rows)
end = time.time()

user_review_time_all = end - start

for row in fetched:
    print(row)


Row(user_id='AHS3BYEXVP4RMJMIDJUDKGFROX7Q', review_time=datetime.datetime(2018, 5, 2, 1, 44, 10, 116000), parent_asin='B00U0Z0IUM', rating=5, title='love smell')


In [28]:
#4
# #reviews by product test
product = reviews.select("parent_asin").limit(1).collect()[0]["parent_asin"]

start = time.time()
rows = session.execute("""
select parent_asin, review_time, user_id, rating, title
from amazon_reviews.all_data
where parent_asin=%s
limit 5
""", (product,))
fetched = list(rows)
end = time.time()

product_review_time_all = end - start

for row in fetched:
    print(row)



Row(parent_asin='B00U0Z0IUM', review_time=datetime.datetime(2019, 1, 28, 21, 47, 51, 596000), user_id='AE422FOZHQIMARXVYUW6W2DGXMBQ', rating=5, title='love')
Row(parent_asin='B00U0Z0IUM', review_time=datetime.datetime(2018, 10, 4, 7, 16, 51, 76000), user_id='AGHOEUGLLBU4J554C6RG6RXL4SLQ', rating=5, title='awesome soap imo people like chemical sensitivity')
Row(parent_asin='B00U0Z0IUM', review_time=datetime.datetime(2018, 5, 14, 23, 35, 43, 523000), user_id='AFQYT7XPMVNOZL57M54M7RRWUMQA', rating=1, title='wonderful scent changed')
Row(parent_asin='B00U0Z0IUM', review_time=datetime.datetime(2018, 5, 8, 2, 23, 53, 742000), user_id='AHG5427H6WYRNG2A7YGCHO6NOPHQ', rating=1, title='new formula waste money')
Row(parent_asin='B00U0Z0IUM', review_time=datetime.datetime(2018, 5, 2, 1, 44, 10, 116000), user_id='AHS3BYEXVP4RMJMIDJUDKGFROX7Q', rating=5, title='love smell')


In [29]:
#5
# products table test (metadata by asin)
test_product = products_sample.select("parent_asin").limit(1).collect()[0]["parent_asin"]

start = time.time()
row = session.execute("""
select parent_asin, product_title, product_description
from amazon_reviews.all_data
where parent_asin=%s
limit 1
""", (test_product,)).one()
end = time.time()

product_time_all = end - start

print(row)

Row(parent_asin='B01CUPMQZE', product_title='howard lc008 leather conditioner 8 ounce 4 pack', product_description='')


In [30]:
#create dataframe to store times for normalised and denormalsied for comparisons
time_df = {
    "Keyword Time (ms)": [keyword_time * 1000, keyword_time_all * 1000],
    "Product Rating Time (ms)": [product_rating_time * 1000, product_rating_time_all * 1000],
    "User Review Time (ms)": [user_review_time * 1000, user_review_time_all * 1000],
    "Product Review Time (ms)": [product_review_time * 1000, product_review_time_all * 1000],
    "Product Meta Data Time (ms)": [product_time * 1000, product_time_all * 1000]
}

#exporting as parquet file for comparsions
time_df = pd.DataFrame(time_df, index=["Normalised", "Denormalised"])
print(time_df)

              Keyword Time (ms)  Product Rating Time (ms)  \
Normalised            61.454058                 19.012690   
Denormalised         109.723568                 16.877413   

              User Review Time (ms)  Product Review Time (ms)  \
Normalised                25.197744                 16.478300   
Denormalised              18.063784                 22.074699   

              Product Meta Data Time (ms)  
Normalised                       8.658409  
Denormalised                    11.496067  


In [33]:
time_df.to_parquet("time_data.parquet")